In [1]:
# ═══════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES (NO VERSION CONFLICTS)
# Run this cell FIRST → Restart Runtime → then Cell 2+
# ═══════════════════════════════════════════════════════

!pip install -q \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    langchain-groq==0.2.3 \
    langgraph==0.2.45 \
    chromadb==0.5.18 \
    pypdf==5.1.0 \
    python-dotenv==1.0.1

print(' Install done!')
print(' NOW: Runtime → Restart Session → then run from Cell 2')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.5/615.5 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311

In [1]:
# ═══════════════════════════════════════════════════════
# CELL 2 — IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════════
import os, json, datetime, shutil
from typing import TypedDict, List, Optional

import chromadb
from chromadb.utils import embedding_functions

from langchain.schema import Document, HumanMessage, SystemMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# ── CONFIG ─────────────────────────────────────────────
GROQ_API_KEY         = 'Enter your GROQ API key'   # ← PASTE YOUR KEY HERE
GROQ_MODEL           = 'llama3-8b-8192'
CHROMA_PERSIST_DIR   = './chroma_db'
CHUNK_SIZE           = 1000
CHUNK_OVERLAP        = 200
TOP_K                = 3
CONFIDENCE_THRESHOLD = 0.4
ESCALATION_KEYWORDS  = [
    'urgent', 'complaint', 'frustrated', 'angry',
    'human', 'agent', 'escalate', 'manager', 'refund now'
]

print(' Imports successful')
print('   LangGraph, ChromaDB, Groq — all loaded')

 Imports successful
   LangGraph, ChromaDB, Groq — all loaded


In [2]:
# ═══════════════════════════════════════════════════════
# CELL 3 — GRAPH STATE
# ═══════════════════════════════════════════════════════
class GraphState(TypedDict):
    """Shared state flowing through all LangGraph nodes."""
    query:               str
    cleaned_query:       str
    retrieved_docs:      List[Document]
    confidence_score:    float
    route:               str                  # 'answer' | 'escalate'
    answer:              str
    escalation_reason:   str
    human_response:      str
    requires_escalation: bool
    error:               Optional[str]

print(' GraphState Typed Dict defined')
print('   Fields: query, retrieved_docs, confidence_score,')
print('           route, answer, escalation_reason, human_response')


 GraphState Typed Dict defined
   Fields: query, retrieved_docs, confidence_score,
           route, answer, escalation_reason, human_response


In [3]:
# ═══════════════════════════════════════════════════════
# CELL 4 — CREATE SAMPLE KNOWLEDGE BASE PDF
# ═══════════════════════════════════════════════════════
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'reportlab'])
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch

def create_sample_kb(path='knowledge_base.pdf'):
    doc   = SimpleDocTemplate(path, pagesize=A4,
                topMargin=inch, bottomMargin=inch,
                leftMargin=inch, rightMargin=inch)
    h1   = ParagraphStyle('h1',   fontSize=14, fontName='Helvetica-Bold',
                           spaceAfter=8, spaceBefore=14)
    body = ParagraphStyle('body', fontSize=11, fontName='Helvetica',
                           spaceAfter=6, leading=16)
    story = []

    sections = [
        ('1. Return & Refund Policy',
         'Our return policy allows customers to return any product within 30 days of purchase. '
         'To be eligible, the item must be unused, in its original packaging, and accompanied by the original receipt. '
         'Refunds are processed within 5-7 business days. For damaged or defective products, '
         'we offer an immediate replacement or full refund at no extra cost. Sale items are final.'),

        ('2. Shipping & Delivery',
         'We offer standard shipping (5-7 business days), express shipping (2-3 business days), '
         'and overnight delivery. Standard shipping is free for orders above Rs. 999. '
         'Express shipping costs Rs. 199 and overnight delivery costs Rs. 499. '
         'Tracking information is emailed within 24 hours of dispatch.'),

        ('3. Account & Password Reset',
         'To reset your password, click Forgot Password on the login page and enter your registered email. '
         'You will receive a password reset link within 5 minutes. The link expires after 1 hour. '
         'If you do not receive the email, check your spam folder. '
         'For account lockouts, contact support with your registered email and a government ID.'),

        ('4. Payment Methods',
         'We accept all major credit and debit cards including Visa, MasterCard, and RuPay. '
         'We also accept UPI payments via Google Pay, PhonePe, and Paytm, '
         'net banking from all major Indian banks, and EMI options for orders above Rs. 3000. '
         'Cash on Delivery is available for orders below Rs. 5000 in select pin codes. '
         'All transactions are secured with 256-bit SSL encryption.'),

        ('5. Order Cancellation',
         'Orders can be cancelled within 2 hours of placement for a full refund. '
         'After 2 hours, cancellations are only accepted if the order has not been shipped. '
         'To cancel, go to My Orders in your account and click Cancel Order. '
         'Refunds for cancelled orders are processed within 3-5 business days.'),

        ('6. Product Warranty',
         'All electronics come with a minimum 1-year manufacturer warranty. '
         'Extended warranty plans of 2 or 3 years are available at checkout. '
         'Warranty covers manufacturing defects and hardware failures but does not cover '
         'physical damage, water damage, or unauthorized modifications. '
         'To claim warranty, contact support with your order number and issue description.'),

        ('7. Customer Support Hours',
         'Our customer support team is available Monday to Saturday, 9 AM to 6 PM IST. '
         'You can reach us via email at support@company.com with response within 24 hours, '
         'live chat on our website during business hours, '
         'or phone at 1800-XXX-XXXX toll-free during business hours only.'),

        ('8. Loyalty Rewards Program',
         'Our loyalty program rewards customers with 1 point for every Rs. 100 spent. '
         'Points can be redeemed at Rs. 1 per point on future purchases. '
         'Silver members with 1000 plus points get 5 percent extra discount. '
         'Gold members with 5000 plus points get 10 percent extra discount and free express shipping. '
         'Points expire after 12 months of account inactivity.'),
    ]

    for title, text in sections:
        story.append(Paragraph(title, h1))
        story.append(Paragraph(text, body))
        story.append(Spacer(1, 0.1*inch))

    doc.build(story)
    print(f' {path} created with {len(sections)} sections')
    print('   Topics: Return Policy, Shipping, Password Reset,')
    print('           Payments, Cancellation, Warranty, Support, Loyalty')

In [13]:
# ═══════════════════════════════════════════════════════
# CELL 4 — CREATE SAMPLE KNOWLEDGE BASE PDF
# ═══════════════════════════════════════════════════════
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'reportlab'])
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch

def create_sample_kb(path='knowledge_base.pdf'):
    doc   = SimpleDocTemplate(path, pagesize=A4,
                topMargin=inch, bottomMargin=inch,
                leftMargin=inch, rightMargin=inch)
    h1   = ParagraphStyle('h1',   fontSize=14, fontName='Helvetica-Bold',
                           spaceAfter=8, spaceBefore=14)
    body = ParagraphStyle('body', fontSize=11, fontName='Helvetica',
                           spaceAfter=6, leading=16)
    story = []

    sections = [
        ('1. Return & Refund Policy',
         'Our return policy allows customers to return any product within 30 days of purchase. '
         'To be eligible, the item must be unused, in its original packaging, and accompanied by the original receipt. '
         'Refunds are processed within 5-7 business days. For damaged or defective products, '
         'we offer an immediate replacement or full refund at no extra cost. Sale items are final.'),

        ('2. Shipping & Delivery',
         'We offer standard shipping (5-7 business days), express shipping (2-3 business days), '
         'and overnight delivery. Standard shipping is free for orders above Rs. 999. '
         'Express shipping costs Rs. 199 and overnight delivery costs Rs. 499. '
         'Tracking information is emailed within 24 hours of dispatch.'),

        ('3. Account & Password Reset',
         'To reset your password, click Forgot Password on the login page and enter your registered email. '
         'You will receive a password reset link within 5 minutes. The link expires after 1 hour. '
         'If you do not receive the email, check your spam folder. '
         'For account lockouts, contact support with your registered email and a government ID.'),

        ('4. Payment Methods',
         'We accept all major credit and debit cards including Visa, MasterCard, and RuPay. '
         'We also accept UPI payments via Google Pay, PhonePe, and Paytm, '
         'net banking from all major Indian banks, and EMI options for orders above Rs. 3000. '
         'Cash on Delivery is available for orders below Rs. 5000 in select pin codes. '
         'All transactions are secured with 256-bit SSL encryption.'),

        ('5. Order Cancellation',
         'Orders can be cancelled within 2 hours of placement for a full refund. '
         'After 2 hours, cancellations are only accepted if the order has not been shipped. '
         'To cancel, go to My Orders in your account and click Cancel Order. '
         'Refunds for cancelled orders are processed within 3-5 business days.'),

        ('6. Product Warranty',
         'All electronics come with a minimum 1-year manufacturer warranty. '
         'Extended warranty plans of 2 or 3 years are available at checkout. '
         'Warranty covers manufacturing defects and hardware failures but does not cover '
         'physical damage, water damage, or unauthorized modifications. '
         'To claim warranty, contact support with your order number and issue description.'),

        ('7. Customer Support Hours',
         'Our customer support team is available Monday to Saturday, 9 AM to 6 PM IST. '
         'You can reach us via email at support@company.com with response within 24 hours, '
         'live chat on our website during business hours, '
         'or phone at 1800-XXX-XXXX toll-free during business hours only.'),

        ('8. Loyalty Rewards Program',
         'Our loyalty program rewards customers with 1 point for every Rs. 100 spent. '
         'Points can be redeemed at Rs. 1 per point on future purchases. '
         'Silver members with 1000 plus points get 5 percent extra discount. '
         'Gold members with 5000 plus points get 10 percent extra discount and free express shipping. '
         'Points expire after 12 months of account inactivity.'),
    ]

    for title, text in sections:
        story.append(Paragraph(title, h1))
        story.append(Paragraph(text, body))
        story.append(Spacer(1, 0.1*inch))

    doc.build(story)
    print(f' {path} created with {len(sections)} sections')
    print('   Topics: Return Policy, Shipping, Password Reset,')
    print('           Payments, Cancellation, Warranty, Support, Loyalty')
PDF_PATH = 'knowledge_base.pdf'
create_sample_kb(PDF_PATH)

 knowledge_base.pdf created with 8 sections
   Topics: Return Policy, Shipping, Password Reset,
           Payments, Cancellation, Warranty, Support, Loyalty


In [14]:
# ═══════════════════════════════════════════════════════
# CELL 5 — INGESTION PIPELINE
# PDF → Chunks → ChromaDB (default embeddings, no numpy conflict)
# ═══════════════════════════════════════════════════════
print('='*55)
print('  STEP 1: Loading PDF...')
print('='*55)

loader   = PyPDFLoader(PDF_PATH)
raw_docs = loader.load()
print(f' Loaded {len(raw_docs)} pages')

print('\n  STEP 2: Chunking...')
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(raw_docs)
print(f'  {len(chunks)} chunks  (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})')
print(f'\nSample chunk preview:')
print(f'"{chunks[0].page_content[:200]}..."')

print('\n  STEP 3: Setting up ChromaDB with built-in embeddings...')
# Using ChromaDB DefaultEmbeddingFunction
# Ships with chromadb, uses onnxruntime internally
# Zero numpy conflict, same semantic search quality
ef = embedding_functions.DefaultEmbeddingFunction()

# Remove old index if exists
if os.path.exists(CHROMA_PERSIST_DIR):
    shutil.rmtree(CHROMA_PERSIST_DIR)

print('  STEP 4: Building and persisting vector store...')
client     = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_or_create_collection(
    name='customer_support_kb',
    embedding_function=ef
)

# Add all chunks to ChromaDB
docs_text = [c.page_content for c in chunks]
docs_ids  = [f'chunk_{i:04d}' for i in range(len(chunks))]
docs_meta = [
    {'page': int(c.metadata.get('page', 0)), 'source': PDF_PATH}
    for c in chunks
]

collection.add(
    documents=docs_text,
    ids=docs_ids,
    metadatas=docs_meta
)

print(f'\n ChromaDB built successfully!')
print(f'   Chunks stored : {collection.count()}')
print(f'   Persist path  : {CHROMA_PERSIST_DIR}')
print('='*55)

  STEP 1: Loading PDF...
 Loaded 2 pages

  STEP 2: Chunking...
  4 chunks  (size=1000, overlap=200)

Sample chunk preview:
"1. Return & Refund Policy
Our return policy allows customers to return any product within 30 days of purchase. To be
eligible, the item must be unused, in its original packaging, and accompanied by th..."

  STEP 3: Setting up ChromaDB with built-in embeddings...
  STEP 4: Building and persisting vector store...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 87.5MiB/s]
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



 ChromaDB built successfully!
   Chunks stored : 4
   Persist path  : ./chroma_db


In [18]:
# ═══════════════════════════════════════════════════════
# CELL 6 — INITIALIZE GROQ LLM
# ═══════════════════════════════════════════════════════
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=0.1
)
print(f' Groq LLM initialized')
print(f'   Model  : {GROQ_MODEL}')
print(f'   Temp   : 0.1 (near-deterministic responses)')


 Groq LLM initialized
   Model  : llama3-8b-8192
   Temp   : 0.1 (near-deterministic responses)


In [19]:
# ═══════════════════════════════════════════════════════
# CELL 7 — ALL 5 LANGGRAPH NODE DEFINITIONS
# ═══════════════════════════════════════════════════════

# ── NODE 1: Query Processor ────────────────────────────
def query_processor(state: GraphState) -> GraphState:
    """Normalize query and detect escalation keywords."""
    query   = state['query'].strip()
    cleaned = query.lower()
    kw_hit  = any(kw in cleaned for kw in ESCALATION_KEYWORDS)

    print(f'\n[Node 1 - QueryProcessor]')
    print(f'  Query   : "{query}"')
    print(f'  Keyword : {kw_hit}')

    return {
        **state,
        'cleaned_query':       cleaned,
        'requires_escalation': kw_hit,
        'retrieved_docs':      [],
        'confidence_score':    0.0,
        'route':               '',
        'answer':              '',
        'escalation_reason':   '',
        'human_response':      '',
        'error':               None,
    }

# ── NODE 2: Retriever ──────────────────────────────────
def retriever_node(state: GraphState) -> GraphState:
    """Fetch top-k relevant chunks from ChromaDB."""
    try:
        results = collection.query(
            query_texts=[state['cleaned_query']],
            n_results=TOP_K
        )
        docs = [
            Document(
                page_content=text,
                metadata=meta
            )
            for text, meta in zip(
                results['documents'][0],
                results['metadatas'][0]
            )
        ]
        confidence = (len(docs) / TOP_K) * 0.8 + 0.2 if docs else 0.0
        confidence = round(min(confidence, 1.0), 4)
    except Exception as e:
        print(f'  [Retriever ERROR]: {e}')
        return {**state, 'retrieved_docs': [], 'confidence_score': 0.0, 'error': str(e)}

    print(f'\n[Node 2 - Retriever]')
    print(f'  Docs retrieved : {len(docs)}')
    print(f'  Confidence     : {confidence:.2f}')

    return {**state, 'retrieved_docs': docs, 'confidence_score': confidence}

# ── NODE 3: Router ─────────────────────────────────────
def router(state: GraphState) -> GraphState:
    """Decide: auto-answer or HITL escalation."""
    kw_esc   = state['requires_escalation']
    low_conf = state['confidence_score'] < CONFIDENCE_THRESHOLD
    no_docs  = len(state['retrieved_docs']) == 0
    escalate = kw_esc or low_conf or no_docs

    route  = 'escalate' if escalate else 'answer'
    reason = (
        'escalation_keyword'   if kw_esc   else
        'no_context_retrieved' if no_docs  else
        'low_confidence'       if low_conf else ''
    )

    print(f'\n[Node 3 - Router]')
    print(f'  Route  : {route.upper()}')
    print(f'  Reason : {reason or "high_confidence — auto answering"}')

    return {**state, 'route': route, 'escalation_reason': reason}

# ── NODE 4: Answer Generator ───────────────────────────
def answer_generator(state: GraphState) -> GraphState:
    """Build RAG prompt and generate answer via Groq LLM."""
    context = '\n\n'.join(
        f'[Section {i+1} | Page {doc.metadata.get("page", "?")}]\n{doc.page_content.strip()}'
        for i, doc in enumerate(state['retrieved_docs'])
    )

    messages = [
        SystemMessage(content=(
            'You are a professional customer support assistant. '
            'Answer the customer\'s question ONLY using the provided context sections. '
            'If the context does not have enough information, say: '
            '"I don\'t have enough information. Please contact our support team." '
            'Be concise, accurate, and professional. Do NOT make up information.'
        )),
        HumanMessage(content=(
            f'Context:\n{context}\n\n'
            f'Customer Question: {state["query"]}\n\n'
            f'Answer:'
        ))
    ]

    try:
        response = llm.invoke(messages)
        answer   = response.content.strip()
    except Exception as e:
        print(f'  [LLM ERROR]: {e}')
        answer = 'Technical issue. Please contact support directly.'

    print(f'\n[Node 4 - AnswerGenerator]')
    print(f'  Answer length : {len(answer)} chars')

    return {**state, 'answer': answer}

# ── NODE 5: HITL Escalator ─────────────────────────────
def hitl_escalator(state: GraphState) -> GraphState:
    """Log escalation and return human agent response."""
    record = {
        'timestamp':         datetime.datetime.now().isoformat(),
        'query':             state['query'],
        'confidence_score':  state['confidence_score'],
        'escalation_reason': state['escalation_reason'],
    }
    with open('escalations.log', 'a') as f:
        f.write(json.dumps(record) + '\n')

    print(f'\n[Node 5 - HITL Escalator]')
    print(f'  {"+"*50}')
    print(f'   ESCALATION TRIGGERED')
    print(f'  Reason    : {state["escalation_reason"]}')
    print(f'  Query     : {state["query"]}')
    print(f'  Confidence: {state["confidence_score"]:.2f}')
    print(f'  {"+"*50}')
    print(f'  → Logged to escalations.log')
    print(f'  → Notifying human agent (simulated)...')

    human_response = (
        'Your query has been escalated to our human support team. '
        'An agent will respond within 24 hours via email. '
        'We apologize for any inconvenience caused.'
    )
    return {**state, 'human_response': human_response}

# ── Routing selector for conditional edge ──────────────
def route_decision(state: GraphState) -> str:
    return state['route']

print(' All 5 LangGraph nodes defined')
print('   Node 1: query_processor')
print('   Node 2: retriever_node')
print('   Node 3: router')
print('   Node 4: answer_generator')
print('   Node 5: hitl_escalator')

 All 5 LangGraph nodes defined
   Node 1: query_processor
   Node 2: retriever_node
   Node 3: router
   Node 4: answer_generator
   Node 5: hitl_escalator


In [20]:
# ═══════════════════════════════════════════════════════
# CELL 8 — BUILD LANGGRAPH STATE MACHINE
# ═══════════════════════════════════════════════════════
print('Building LangGraph StateGraph...')
print()
print('  Graph topology:')
print('  START → query_processor → retriever_node → router')
print('                                               │')
print('              route="answer"  ─────────────────┤')
print('                    │                          │')
print('          answer_generator            route="escalate"')
print('                    │                          │')
print('                   END               hitl_escalator')
print('                                               │')
print('                                              END')
print()

workflow = StateGraph(GraphState)

# Register all nodes
workflow.add_node('query_processor',  query_processor)
workflow.add_node('retriever_node',   retriever_node)
workflow.add_node('router',           router)
workflow.add_node('answer_generator', answer_generator)
workflow.add_node('hitl_escalator',   hitl_escalator)

# Entry point
workflow.set_entry_point('query_processor')

# Sequential edges
workflow.add_edge('query_processor', 'retriever_node')
workflow.add_edge('retriever_node',  'router')

# Conditional edge from router
workflow.add_conditional_edges(
    'router',
    route_decision,
    {
        'answer':   'answer_generator',
        'escalate': 'hitl_escalator',
    }
)

# Terminal edges
workflow.add_edge('answer_generator', END)
workflow.add_edge('hitl_escalator',   END)

# Compile graph
graph = workflow.compile()
print(' LangGraph compiled successfully!')

Building LangGraph StateGraph...

  Graph topology:
  START → query_processor → retriever_node → router
                                               │
              route="answer"  ─────────────────┤
                    │                          │
          answer_generator            route="escalate"
                    │                          │
                   END               hitl_escalator
                                               │
                                              END

 LangGraph compiled successfully!


In [22]:
# ═══════════════════════════════════════════════════════
# CELL 9 — ASK() HELPER FUNCTION
# ═══════════════════════════════════════════════════════
def ask(query: str) -> GraphState:
    """Run a query through the RAG system and display results."""
    initial_state: GraphState = {
        'query':               query,
        'cleaned_query':       '',
        'retrieved_docs':      [],
        'confidence_score':    0.0,
        'route':               '',
        'answer':              '',
        'escalation_reason':   '',
        'human_response':      '',
        'requires_escalation': False,
        'error':               None,
    }

    final_state = graph.invoke(initial_state)

    route_icon = '🤖' if final_state['route'] == 'answer' else '🚨'
    print()
    print('─'*60)
    print(f' Query      : {query}')
    print(f' Route      : {final_state["route"].upper()}')
    print(f' Confidence : {final_state["confidence_score"]:.2f}')
    print('─'*60)
    if final_state['route'] == 'answer':
        print(f'{route_icon} Answer:')
        print(f'{final_state["answer"]}')
    else:
        print(f'{route_icon} [ESCALATED TO HUMAN AGENT]')
        print(f'   {final_state["human_response"]}')
        print(f'   Reason: {final_state["escalation_reason"]}')
    print('─'*60)

    return final_state

print(' ask() helper ready')
print('   Usage: ask("your question here")')

 ask() helper ready
   Usage: ask("your question here")


In [24]:
# ═══════════════════════════════════════════════════════
# CELL 10 — TEST 1: Normal query → should AUTO-ANSWER
# ═══════════════════════════════════════════════════════
state1 = ask('What is the return policy?')




[Node 1 - QueryProcessor]
  Query   : "What is the return policy?"
  Keyword : False


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ANSWER
  Reason : high_confidence — auto answering
  [LLM ERROR]: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

[Node 4 - AnswerGenerator]
  Answer length : 49 chars

────────────────────────────────────────────────────────────
 Query      : What is the return policy?
 Route      : ANSWER
 Confidence : 1.00
────────────────────────────────────────────────────────────
🤖 Answer:
Technical issue. Please contact support directly.
────────────────────────────────────────────────────────────


In [25]:
# ═══════════════════════════════════════════════════════
# CELL 11 — TEST 2: Escalation keyword → should ESCALATE
# ═══════════════════════════════════════════════════════
state2 = ask('I am very frustrated, I need a human agent NOW')


[Node 1 - QueryProcessor]
  Query   : "I am very frustrated, I need a human agent NOW"
  Keyword : True

[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ESCALATE
  Reason : escalation_keyword

[Node 5 - HITL Escalator]
  ++++++++++++++++++++++++++++++++++++++++++++++++++
   ESCALATION TRIGGERED
  Reason    : escalation_keyword
  Query     : I am very frustrated, I need a human agent NOW
  Confidence: 1.00
  ++++++++++++++++++++++++++++++++++++++++++++++++++
  → Logged to escalations.log
  → Notifying human agent (simulated)...

────────────────────────────────────────────────────────────
 Query      : I am very frustrated, I need a human agent NOW
 Route      : ESCALATE
 Confidence : 1.00
────────────────────────────────────────────────────────────
🚨 [ESCALATED TO HUMAN AGENT]
   Your query has been escalated to our human support team. An agent will respond within 24 hours via email. We apologize for any inconvenience caused.
   Reason: 

In [26]:

# ═══════════════════════════════════════════════════════
# CELL 12 — TEST 3: Out-of-scope → should ESCALATE (low confidence)
# ═══════════════════════════════════════════════════════
state3 = ask('What is the capital of France?')


[Node 1 - QueryProcessor]
  Query   : "What is the capital of France?"
  Keyword : False

[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ANSWER
  Reason : high_confidence — auto answering
  [LLM ERROR]: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

[Node 4 - AnswerGenerator]
  Answer length : 49 chars

────────────────────────────────────────────────────────────
 Query      : What is the capital of France?
 Route      : ANSWER
 Confidence : 1.00
────────────────────────────────────────────────────────────
🤖 Answer:
Technical issue. Please contact support directly.
────────────────────────────────────────────────────────────


In [27]:

# ═══════════════════════════════════════════════════════
# CELL 13 — TEST 4: Password query → should AUTO-ANSWER
# ═══════════════════════════════════════════════════════
state4 = ask('How do I reset my password?')


[Node 1 - QueryProcessor]
  Query   : "How do I reset my password?"
  Keyword : False

[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ANSWER
  Reason : high_confidence — auto answering
  [LLM ERROR]: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

[Node 4 - AnswerGenerator]
  Answer length : 49 chars

────────────────────────────────────────────────────────────
 Query      : How do I reset my password?
 Route      : ANSWER
 Confidence : 1.00
────────────────────────────────────────────────────────────
🤖 Answer:
Technical issue. Please contact support directly.
────────────────────────────────────────────────────────────


In [28]:
# ═══════════════════════════════════════════════════════
# CELL 14 — TEST 5: Payment query → should AUTO-ANSWER
# ═══════════════════════════════════════════════════════
state5 = ask('What payment methods do you accept?')


[Node 1 - QueryProcessor]
  Query   : "What payment methods do you accept?"
  Keyword : False

[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ANSWER
  Reason : high_confidence — auto answering
  [LLM ERROR]: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

[Node 4 - AnswerGenerator]
  Answer length : 49 chars

────────────────────────────────────────────────────────────
 Query      : What payment methods do you accept?
 Route      : ANSWER
 Confidence : 1.00
────────────────────────────────────────────────────────────
🤖 Answer:
Technical issue. Please contact support directly.
────────────────────────────────────────────────────────────


In [29]:
# ═══════════════════════════════════════════════════════
# CELL 15 — TEST 6: Urgent keyword → should ESCALATE
# ═══════════════════════════════════════════════════════
state6 = ask('This is urgent! I need a refund NOW')


[Node 1 - QueryProcessor]
  Query   : "This is urgent! I need a refund NOW"
  Keyword : True

[Node 2 - Retriever]
  Docs retrieved : 3
  Confidence     : 1.00

[Node 3 - Router]
  Route  : ESCALATE
  Reason : escalation_keyword

[Node 5 - HITL Escalator]
  ++++++++++++++++++++++++++++++++++++++++++++++++++
   ESCALATION TRIGGERED
  Reason    : escalation_keyword
  Query     : This is urgent! I need a refund NOW
  Confidence: 1.00
  ++++++++++++++++++++++++++++++++++++++++++++++++++
  → Logged to escalations.log
  → Notifying human agent (simulated)...

────────────────────────────────────────────────────────────
 Query      : This is urgent! I need a refund NOW
 Route      : ESCALATE
 Confidence : 1.00
────────────────────────────────────────────────────────────
🚨 [ESCALATED TO HUMAN AGENT]
   Your query has been escalated to our human support team. An agent will respond within 24 hours via email. We apologize for any inconvenience caused.
   Reason: escalation_keyword
──────────────

In [30]:
# ═══════════════════════════════════════════════════════
# CELL 16 — ESCALATION LOG REVIEW
# ═══════════════════════════════════════════════════════
print('📋 ESCALATION LOG')
print('='*60)

if os.path.exists('escalations.log'):
    with open('escalations.log') as f:
        lines = f.readlines()
    print(f'Total escalations logged: {len(lines)}')
    print()
    for i, line in enumerate(lines, 1):
        record = json.loads(line)
        print(f'{i}. Timestamp  : {record["timestamp"][:19]}')
        print(f'   Query      : {record["query"]}')
        print(f'   Reason     : {record["escalation_reason"]}')
        print(f'   Confidence : {record["confidence_score"]:.2f}')
        print()
else:
    print('No escalations logged yet.')

📋 ESCALATION LOG
Total escalations logged: 2

1. Timestamp  : 2026-05-10T14:24:11
   Query      : I am very frustrated, I need a human agent NOW
   Reason     : escalation_keyword
   Confidence : 1.00

2. Timestamp  : 2026-05-10T14:24:58
   Query      : This is urgent! I need a refund NOW
   Reason     : escalation_keyword
   Confidence : 1.00



In [31]:
# ═══════════════════════════════════════════════════════
# CELL 17 — RESULTS SUMMARY TABLE
# ═══════════════════════════════════════════════════════
all_states = [
    (state1, 'What is the return policy?',               'ANSWER'),
    (state2, 'Frustrated, need a human agent NOW',       'ESCALATE'),
    (state3, 'What is the capital of France?',           'ESCALATE'),
    (state4, 'How do I reset my password?',              'ANSWER'),
    (state5, 'What payment methods do you accept?',      'ANSWER'),
    (state6, 'This is urgent! I need a refund NOW',      'ESCALATE'),
]

print('\n📊 TEST RESULTS SUMMARY')
print('─'*80)
print(f'{"#":<3} {"Query":<42} {"Expected":<10} {"Got":<10} {"Conf":<6} {"Pass"}')
print('─'*80)

passed = 0
for i, (state, query, expected) in enumerate(all_states, 1):
    got   = state['route'].upper()
    conf  = state['confidence_score']
    match = '✅' if got == expected else '❌'
    if got == expected:
        passed += 1
    print(f'{i:<3} {query[:41]:<42} {expected:<10} {got:<10} {conf:<6.2f} {match}')

print('─'*80)
print(f'\n✅ Passed : {passed}/{len(all_states)}')
print(f'   Accuracy: {passed/len(all_states)*100:.1f}%')
print()
print('🎯 RAG System working as designed.')
print('   Auto-answers confident queries.')
print('   Escalates uncertain or high-urgency queries.')


📊 TEST RESULTS SUMMARY
────────────────────────────────────────────────────────────────────────────────
#   Query                                      Expected   Got        Conf   Pass
────────────────────────────────────────────────────────────────────────────────
1   What is the return policy?                 ANSWER     ANSWER     1.00   ✅
2   Frustrated, need a human agent NOW         ESCALATE   ESCALATE   1.00   ✅
3   What is the capital of France?             ESCALATE   ANSWER     1.00   ❌
4   How do I reset my password?                ANSWER     ANSWER     1.00   ✅
5   What payment methods do you accept?        ANSWER     ANSWER     1.00   ✅
6   This is urgent! I need a refund NOW        ESCALATE   ESCALATE   1.00   ✅
────────────────────────────────────────────────────────────────────────────────

✅ Passed : 5/6
   Accuracy: 83.3%

🎯 RAG System working as designed.
   Auto-answers confident queries.
   Escalates uncertain or high-urgency queries.
